# MeSA 2.0 — Train YOLOv8n on the medication dataset (VIS-004)

Run on **Google Colab (free GPU)**: Runtime → Change runtime type → **T4 GPU**.

**Goal (M2):** export `best.pt` with **mAP@50 ≥ 0.90** on the val set.

Steps: install → pull dataset from Roboflow → train → validate → download `best.pt`.

In [ ]:
!nvidia-smi
!pip -q install ultralytics==8.2.103 roboflow

In [ ]:
# Pull the annotated dataset (VIS-003). Get this snippet from Roboflow > Export > YOLOv8.
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("mesa-medication")
dataset = project.version(1).download("yolov8")
print(dataset.location)  # contains data.yaml

In [ ]:
# Fine-tune YOLOv8n. 100 epochs is plenty for ~600 imgs / 6 classes; watch for overfit.
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,        # early stop if val mAP plateaus
    project="mesa",
    name="yolov8n_med",
)

In [ ]:
# Validate — check mAP@50 >= 0.90 (M2 gate).
metrics = model.val()
print("mAP@50   :", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

In [ ]:
# Download best.pt -> commit to repo as models/best.pt (gitignored; store via release/LFS if large).
from google.colab import files
best = "mesa/yolov8n_med/weights/best.pt"
files.download(best)

If mAP@50 < 0.90: collect hard negatives (VIS-007), re-upload to Roboflow, bump the
dataset version, and re-run. Copy final metrics into `docs/eval-report-template.md`.